[Guide Index](README.md)


# 03a SPARQL 1.2 in queries

The parts of a query that are specific to SPARQL 1.2: triple functions such as `TRIPLE` and `isTRIPLE`, triple-term and annotation syntax, language and direction functions and the `VERSION` directive. The sections share one example graph, built in the first. See [SPARQL](03-sparql.ipynb) for running a query and reading its result.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.


In [1]:
from starlayer.graph import StarLayerGraph

EX = "http://example.org/"
RDF = "http://www.w3.org/1999/02/22-rdf-syntax-ns#"

## 1. The example graph

The sections of this guide query one graph, built here. It has a name in two languages with a base direction, and the reifications and annotations that the later sections query. It uses the `ex` and `rdf` prefixes..


In [2]:
g_parsed = StarLayerGraph()
g_parsed.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

    # language-direction literals: AcmeCorp's name in English and Arabic
    ex:AcmeCorp ex:name "Acme Corporation"@en--ltr .
    ex:AcmeCorp ex:name "شركة أكمي"@ar--rtl .

    # canonical reification with rdf:reifies
    ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> ;
      ex:source ex:HRSystem ;
      ex:confidence "high" .

    # anonymous inline annotation block (asserted triple)
    ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} .

    # named reifier with annotations
    ex:alice ex:mentions ex:GlobalTech ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:AuditSystem |} .

    # named reifier without annotation block
    ex:alice ex:worksWith ex:SalesTeam ~ ex:stmt2 .
    ex:stmt2 ex:asReported ex:HRSystem .
    ex:stmt2 ex:reportedDate "1/1/2026" .

    # reification shorthand << s p o >> (no parens): an anonymous, unasserted
    # reifier of the same base triple ex:claim reifies above - a triple
    # can have more than one independent reification
    << ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:LinkedIn .

    # unasserted quoted triple term
    ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
''', format='turtle')
print("parsed triples:", len(g_parsed))
print()
# pretty=True is StarLayer's pretty Turtle writer (grouped subjects, {| |} annotation blocks);
# the default serialize() is Oxigraph's flat one-statement-per-line Turtle.
print(g_parsed.serialize(format="turtle", pretty=True))

parsed triples: 20

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:AcmeCorp ex:name "Acme Corporation"@en--ltr, "شركة أكمي"@ar--rtl .

ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} ;
    ex:mentions ex:GlobalTech ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:AuditSystem |} ;
    ex:worksWith ex:SalesTeam ~ ex:stmt2 {| ex:asReported ex:HRSystem ; ex:reportedDate "1/1/2026" |} .

<< ex:alice ex:worksFor ex:AcmeCorp ~ ex:claim >> ex:confidence "high" ;
    ex:source ex:HRSystem .

<< ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:LinkedIn .



## 2. Triple functions

SPARQL 1.2 introduces now functions for working with triple terms.

- `isTRIPLE(x)` — true when a value is a triple term
- `TRIPLE(s, p, o)` — a functional way to write a triple term `<<( s p o )>>`
- `SUBJECT()`, `PREDICATE()`, `OBJECT()` — extract the parts out of a triple term

In [3]:
# 2.1 detect triple-term values using isTRIPLE
print("2.1 isTRIPLE()")
rows = g_parsed.query("""
SELECT ?claim ?p ?statement WHERE {
  ?claim ?p ?statement .
  FILTER( isTRIPLE(?statement))
  FILTER( ?p != rdf:reifies)
}
""")

# anonymous reifiers have a new label on every parse, so sort the printed lines
for line in sorted(
    f"{g_parsed.qname_term(row.claim)} {g_parsed.qname_term(row.p)} {g_parsed.qname_term(row.statement)}"
    for row in rows
):
    print(line)


2.1 isTRIPLE()
ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>>


In [4]:
# 2.2 TRIPLE() and SUBJECT()/PREDICATE()/OBJECT()
print("2.2 TRIPLE()/SUBJECT()/PREDICATE()/OBJECT()")
rows = g_parsed.query("""
SELECT ?s ?p ?o WHERE {
  ex:claim rdf:reifies ?t .
  FILTER(?t = TRIPLE(ex:alice, ex:worksFor, ex:AcmeCorp))
  BIND(SUBJECT(?t) AS ?s)
  BIND(PREDICATE(?t) AS ?p)
  BIND(OBJECT(?t) AS ?o)
}
""")
for row in rows:
    print(g_parsed.qname_term(row.s), g_parsed.qname_term(row.p), g_parsed.qname_term(row.o))


2.2 TRIPLE()/SUBJECT()/PREDICATE()/OBJECT()
ex:alice ex:worksFor ex:AcmeCorp


## 3. Triple-term and annotation syntax in queries

The Turtle 1.2 syntax used to parse and serialize graphs also works in a SPARQL 1.2 query.

`<<( s p o )>>` can be used to represent a triple term.  The shorthand versions `{| ?pred ?val |}`, `~ ?r` and `<< s p o >>`, which stands for the `rdf:reifies` pattern can also be used.  

In [5]:
# 3.1 <<( ?s ?p ?o )>>: bind all terms inside a triple term
print("3.1 <<( ?s ?p ?o )>> pattern")
rows = g_parsed.query("""
SELECT ?claim ?s ?p ?o ?source WHERE {
  ?claim rdf:reifies <<( ?s ?p ?o )>> .
  ?claim ex:source ?source .
  FILTER(?p = ex:worksFor)
}
ORDER BY ?claim ?s ?o
""")

# ?claim can be an anonymous reifier (a BlankNode) as well as a named one (a NamedNode);
# qname_term() writes either: a prefixed name for an IRI, _:label for a blank node.
for row in rows:
    print(
        g_parsed.qname_term(row.claim),
        g_parsed.qname_term(row.s),
        g_parsed.qname_term(row.p),
        g_parsed.qname_term(row.o),
        "Source:", g_parsed.qname_term(row.source),
    )

3.1 <<( ?s ?p ?o )>> pattern
_:c6adff0bcf819035fbe043c47b367f6e ex:alice ex:worksFor ex:AcmeCorp Source: ex:LinkedIn
ex:claim ex:alice ex:worksFor ex:AcmeCorp Source: ex:HRSystem


In [6]:
# 3.2 <<( s p o )>> can also be written as a value in an expression such as BIND(),
# not only as the object of a triple pattern. It is equivalent to TRIPLE(s, p, o) in section 2.
print()
print("3.2 <<( s p o )>> in BIND()")
rows = g_parsed.query("""
SELECT ?tt WHERE {
  BIND( <<( ex:alice ex:worksFor ex:AcmeCorp )>> AS ?tt )
}
""")
for row in rows:
    # a pyoxigraph Triple: a triple term is a value of its own, with .subject/.predicate/.object
    print(type(row.tt).__name__, row.tt)


3.2 <<( s p o )>> in BIND()
Triple <http://example.org/alice> <http://example.org/worksFor> <http://example.org/AcmeCorp>


In [7]:
# 3.3 {| ?pred ?val |}: query an anonymous reifier's annotations inline (asserted triple)
# The reifier also has an rdf:reifies triple, which the FILTER leaves out.
print("3.3 {| ?pred ?val |}")
rows = g_parsed.query("""
SELECT ?pred ?val WHERE {
  ex:alice ex:likes ex:ProductABC {| ?pred ?val |}
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname_term(row.pred), g_parsed.qname_term(row.val))

# 3.4 ~ ?r: bind the reifier itself for a named reifier
print()
print("3.4 ~ ?r")
rows = g_parsed.query("""
SELECT ?r WHERE {
  ex:alice ex:worksWith ex:SalesTeam ~ ?r
}
""")
for row in rows:
    print(g_parsed.qname_term(row.r))

# 3.5 << s p o >> ?pred ?val: reification shorthand, no assertion required
# No FILTER here, so the rdf:reifies triple of the reifier is in the answer.
print()
print("3.5 << s p o >>")
rows = g_parsed.query("""
SELECT ?pred ?val WHERE {
  << ex:alice ex:likes ex:ProductABC >> ?pred ?val
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname_term(row.pred), g_parsed.qname_term(row.val))

3.3 {| ?pred ?val |}
ex:since "2020"
ex:source ex:CRM

3.4 ~ ?r
ex:stmt2

3.5 << s p o >>
ex:since "2020"
ex:source ex:CRM
rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>>


## 4. Language and direction functions

- `LANG()`, `hasLANG()` — read and check a literal's language tag; `LANG()` is a SPARQL 1.1 function, now also aware of direction-tagged literals
- `LANGDIR()`, `hasLANGDIR()` — read and check a literal's base direction
- `STRLANGDIR()` — build a direction-tagged literal

In [8]:
# 4.1 LANG() / hasLANG() / LANGDIR() / hasLANGDIR() over AcmeCorp's two names
print("4.1 LANG()/hasLANG()/LANGDIR()/hasLANGDIR()")
rows = g_parsed.query("""
SELECT ?s ?lang ?hasLang ?dir ?hasDir WHERE {
  ?s ex:name ?lit .
  BIND(LANG(?lit) AS ?lang)
  BIND(hasLANG(?lit) AS ?hasLang)
  BIND(LANGDIR(?lit) AS ?dir)
  BIND(hasLANGDIR(?lit) AS ?hasDir)
}
ORDER BY ?s ?lang
""")
for row in rows:
    # row.lang and row.dir are Literals; .value is the Python str
    print(g_parsed.qname_term(row.s), row.lang.value, row.hasLang.value, row.dir.value, row.hasDir.value)

# 4.2 STRLANGDIR() constructs a direction-tagged literal directly from plain strings
print()
print("4.2 STRLANGDIR()")
rows = g_parsed.query("""
SELECT ?lit WHERE { BIND(STRLANGDIR("hi", "en", "ltr") AS ?lit) }
""")
for row in rows:
    print(row.lit, "| language:", row.lit.language, "| direction:", row.lit.direction)


4.1 LANG()/hasLANG()/LANGDIR()/hasLANGDIR()
ex:AcmeCorp ar true rtl true
ex:AcmeCorp en true ltr true

4.2 STRLANGDIR()
"hi"@en--ltr | language: en | direction: ltr


## 5. The `VERSION "1.2"` query prologue directive

Queries are often stored, shared, or sent to other stores, and not every store supports SPARQL 1.2. A leading `VERSION` line (before any `PREFIX` or `SELECT`) states which version of SPARQL a query needs. It is optional.

- A query that is limited to run on a SPARQL 1.1 store can declare `VERSION "1.1"`. If the query is later edited to add a SPARQL 1.2 feature to it, such as a triple term, StarLayer rejects the query with a `ValueError` when you run it, so the mistake is found in StarLayer and not when the store rejects it.
- A query that relies on SPARQL 1.2 can declare `VERSION "1.2"`, so the requirement can be included in the query itself.

A query with no `VERSION` line is not checked.

StarLayer rejects a query whose `VERSION` line is contrqadicted by its text or that names an unknown version, and it does so before anything is sent to a store. A graph backed by an HTTP endpoint (see the [backend guide](07.02-backend-graph-databases.ipynb)) checks the `VERSION` line before anything is sent. What a remote store does with a `VERSION` line is controlld by that that store. 

In [9]:
# 5.1 VERSION "1.2" declared, and the query does use RDF 1.2 syntax - accepted.
# ?stmt can match an anonymous reifier (a BlankNode) as well as a named one,
# so print via qname_term(), which handles either.
print("5.1 VERSION \"1.2\" matches usage")
rows = g_parsed.query("""
VERSION "1.2"
SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> }
""")
print(sorted(g_parsed.qname_term(row.stmt) for row in rows))

# 5.2 VERSION "1.1" declared, but the query text itself uses a triple term - rejected.
# The check is StarLayer's and runs before the query is run or sent anywhere.
print()
print("5.2 VERSION \"1.1\" mismatch is rejected")
try:
    g_parsed.query("""
    VERSION "1.1"
    SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> }
    """)
except ValueError as error:
    print(type(error).__name__ + ":", error)


5.1 VERSION "1.2" matches usage
['_:c6adff0bcf819035fbe043c47b367f6e', 'ex:claim']

5.2 VERSION "1.1" mismatch is rejected
SPARQL12ConformanceError: SPARQL query declares VERSION '1.1' but uses a triple term, which '1.1' conformance excludes (SPARQL 1.2 Query sec 4.3)
